### Ejercicio 0, cargar y mostrar los datasets como dataframes y sus schemas

In [0]:
movie = spark.read.csv("/Workspace/TallerSparkUVa/Netflix_Dataset_Movie.csv", header=True, inferSchema=True)
movie.show()
movie.printSchema()

+--------+----+--------------------+
|Movie_ID|Year|                Name|
+--------+----+--------------------+
|       1|2003|     Dinosaur Planet|
|       2|2004|Isle of Man TT 20...|
|       3|1997|           Character|
|       4|1994|Paula Abdul's Get...|
|       5|2004|The Rise and Fall...|
|       6|1997|                Sick|
|       7|1992|               8 Man|
|       8|2004|What the #$*! Do ...|
|       9|1991|Class of Nuke 'Em...|
|      10|2001|             Fighter|
|      11|1999|Full Frame: Docum...|
|      12|1947|My Favorite Brunette|
|      13|2003|Lord of the Rings...|
|      14|1982|  Nature: Antarctica|
|      15|1988|Neil Diamond: Gre...|
|      16|1996|           Screamers|
|      17|2005|           7 Seconds|
|      18|1994|    Immortal Beloved|
|      19|2000|By Dawn's Early L...|
|      20|1972|     Seeta Aur Geeta|
+--------+----+--------------------+
only showing top 20 rows
root
 |-- Movie_ID: integer (nullable = true)
 |-- Year: integer (nullable = true)
 |--

In [0]:
rating = spark.read.csv("/Workspace/TallerSparkUVa/Netflix_Dataset_Rating.csv", header=True, inferSchema=True)
rating.show()
rating.printSchema()

+-------+------+--------+
|User_ID|Rating|Movie_ID|
+-------+------+--------+
| 712664|     5|       3|
|1331154|     4|       3|
|2632461|     3|       3|
|  44937|     5|       3|
| 656399|     4|       3|
| 439011|     1|       3|
|1644750|     3|       3|
|2031561|     4|       3|
| 616720|     4|       3|
|2467008|     4|       3|
| 701730|     2|       3|
|1614320|     4|       3|
| 115498|     3|       3|
| 931626|     2|       3|
| 699878|     4|       3|
|1694958|     3|       3|
|  66414|     5|       3|
|2519847|     5|       3|
| 948069|     3|       3|
|  67315|     4|       3|
+-------+------+--------+
only showing top 20 rows
root
 |-- User_ID: integer (nullable = true)
 |-- Rating: integer (nullable = true)
 |-- Movie_ID: integer (nullable = true)



### Ejercicio 1, contar el numero de registros de cada uno de los DataFrames

In [0]:
movie.count()

17770

In [0]:
rating.count()

17337458

### Ejercicio 2, añadir a cada pelicula la puntuación que le ha dado cada usuario


In [0]:
movie_rate = movie.join(rating, "Movie_ID", "inner")
movie_rate.show()

+--------+----+---------+-------+------+
|Movie_ID|Year|     Name|User_ID|Rating|
+--------+----+---------+-------+------+
|       3|1997|Character| 712664|     5|
|       3|1997|Character|1331154|     4|
|       3|1997|Character|2632461|     3|
|       3|1997|Character|  44937|     5|
|       3|1997|Character| 656399|     4|
|       3|1997|Character| 439011|     1|
|       3|1997|Character|1644750|     3|
|       3|1997|Character|2031561|     4|
|       3|1997|Character| 616720|     4|
|       3|1997|Character|2467008|     4|
|       3|1997|Character| 701730|     2|
|       3|1997|Character|1614320|     4|
|       3|1997|Character| 115498|     3|
|       3|1997|Character| 931626|     2|
|       3|1997|Character| 699878|     4|
|       3|1997|Character|1694958|     3|
|       3|1997|Character|  66414|     5|
|       3|1997|Character|2519847|     5|
|       3|1997|Character| 948069|     3|
|       3|1997|Character|  67315|     4|
+--------+----+---------+-------+------+
only showing top

### Ejercicio 3, Asignar a cada registro de película su nota media

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import avg

wind = Window.partitionBy("Movie_ID")

avg_score= movie_rate.withColumn("avg_score", avg("Rating").over(wind))

avg_score.show()


+--------+----+---------+-------+------+------------------+
|Movie_ID|Year|     Name|User_ID|Rating|         avg_score|
+--------+----+---------+-------+------+------------------+
|       3|1997|Character| 712664|     5|3.6213910761154855|
|       3|1997|Character|1331154|     4|3.6213910761154855|
|       3|1997|Character|2632461|     3|3.6213910761154855|
|       3|1997|Character|  44937|     5|3.6213910761154855|
|       3|1997|Character| 656399|     4|3.6213910761154855|
|       3|1997|Character| 439011|     1|3.6213910761154855|
|       3|1997|Character|1644750|     3|3.6213910761154855|
|       3|1997|Character|2031561|     4|3.6213910761154855|
|       3|1997|Character| 616720|     4|3.6213910761154855|
|       3|1997|Character|2467008|     4|3.6213910761154855|
|       3|1997|Character| 701730|     2|3.6213910761154855|
|       3|1997|Character|1614320|     4|3.6213910761154855|
|       3|1997|Character| 115498|     3|3.6213910761154855|
|       3|1997|Character| 931626|     2|

### Ejercicio 4, tener solo 1 registro por movie_ID donde podamos ver su avg_score

In [0]:
movie_avg_score_clean = avg_score.drop("User_ID","Rating").dropDuplicates()
movie_avg_score_clean.show(truncate=False)


+--------+----+---------------------------------+------------------+
|Movie_ID|Year|Name                             |avg_score         |
+--------+----+---------------------------------+------------------+
|3       |1997|Character                        |3.6213910761154855|
|18      |1994|Immortal Beloved                 |3.7675974094914535|
|46      |1964|Rudolph the Red-Nosed Reindeer   |3.7612414036325164|
|83      |1983|Silkwood                         |3.7179717432950192|
|110     |1989|Scandal                          |3.034353193773484 |
|122     |2002|Cube 2: Hypercube                |2.9024563901744393|
|171     |1957|Funny Face                       |3.755378338278932 |
|213     |2001|Dinner Rush                      |3.446800157047507 |
|215     |1998|That '70s Show: Season 1         |4.036098923015556 |
|216     |2000|Impostor                         |3.216482649842271 |
|241     |1959|North by Northwest               |4.1432215743440235|
|256     |2000|Ghost Dog: The Way 

In [0]:
movie_avg_score_clean.count()

1350

### Ejercicio 5, Crear una nueva columna categórica (de texto) que se rija por las siguientes normas
- Para puntuaciones medias inferiores a 2,5, Low
- Para mayores de 3, Medium
- Para mayores de 4 High

In [0]:
from pyspark.sql.functions import col, when

movie_with_category = movie_avg_score_clean.withColumn(
    "Category",
    when(col("avg_score") < 2.5, "Low")
            .when(col("avg_score") < 4, "Medium")
            .otherwise("High")
)

movie_with_category.show(truncate=False)


+--------+----+---------------------------------+------------------+--------+
|Movie_ID|Year|Name                             |avg_score         |Category|
+--------+----+---------------------------------+------------------+--------+
|3       |1997|Character                        |3.6213910761154855|Medium  |
|18      |1994|Immortal Beloved                 |3.7675974094914535|Medium  |
|46      |1964|Rudolph the Red-Nosed Reindeer   |3.7612414036325164|Medium  |
|83      |1983|Silkwood                         |3.7179717432950192|Medium  |
|110     |1989|Scandal                          |3.034353193773484 |Medium  |
|122     |2002|Cube 2: Hypercube                |2.9024563901744393|Medium  |
|171     |1957|Funny Face                       |3.755378338278932 |Medium  |
|213     |2001|Dinner Rush                      |3.446800157047507 |Medium  |
|215     |1998|That '70s Show: Season 1         |4.036098923015556 |High    |
|216     |2000|Impostor                         |3.2164826498422

### Ejercicio 6, eliminar aquellas películas que tengan menos de un 2.5 de nota

In [0]:
movie_good_score = movie_with_category.filter(col("avg_score") > 2.5)
movie_good_score.count()


1330

### Ejercicio 7, truncar la columna avg_score a 2 decimales 

In [0]:
from pyspark.sql.functions import round

score_rounded = movie_avg_score_clean.withColumn(
    "avg_score",
    round("avg_score", 2)
)
score_rounded.show(truncate=False)

+--------+----+---------------------------------+---------+
|Movie_ID|Year|Name                             |avg_score|
+--------+----+---------------------------------+---------+
|3       |1997|Character                        |3.62     |
|18      |1994|Immortal Beloved                 |3.77     |
|46      |1964|Rudolph the Red-Nosed Reindeer   |3.76     |
|83      |1983|Silkwood                         |3.72     |
|110     |1989|Scandal                          |3.03     |
|122     |2002|Cube 2: Hypercube                |2.9      |
|171     |1957|Funny Face                       |3.76     |
|213     |2001|Dinner Rush                      |3.45     |
|215     |1998|That '70s Show: Season 1         |4.04     |
|216     |2000|Impostor                         |3.22     |
|241     |1959|North by Northwest               |4.14     |
|256     |2000|Ghost Dog: The Way of the Samurai|3.42     |
|289     |1998|The Avengers                     |2.32     |
|334     |2005|The Pacifier             

### Ejercicio 8, Obtener el top 10 de películas mejor valoradas y mostrar los registros con el nombre completo

In [0]:
top_10_movies = movie_good_score.orderBy(col("avg_score"), ascending=False).limit(10)
top_10_movies.show()

+--------+----+--------------------+-----------------+--------+
|Movie_ID|Year|                Name|        avg_score|Category|
+--------+----+--------------------+-----------------+--------+
|    3456|2004|      Lost: Season 1|4.665432098765432|    High|
|    2102|1994|The Simpsons: Sea...|4.589824034920202|    High|
|    3444|2004|Family Guy: Freak...|4.520766378244747|    High|
|    1476|2004|Six Feet Under: S...|4.461601211979955|    High|
|    4238|2000|           Inu-Yasha|4.457773512476008|    High|
|    2568|2004|Stargate SG-1: Se...|4.456026058631922|    High|
|    1256|1994|The Best of Frien...|4.449167996352861|    High|
|    4427|2001|The West Wing: Se...| 4.43625843780135|    High|
|    2452|2001|Lord of the Rings...| 4.43148917942777|    High|
|    1947|2002|Gilmore Girls: Se...|4.428942582488959|    High|
+--------+----+--------------------+-----------------+--------+



### Ejercicio 9, Escribir la tabla como parquet con year como columna de partición

**Notas:**

Recordemos que para poder tener nuestro propio espacio deberemos crear un catalog (si no queremos complicarnos podríamos usar el default de databricks). Para ello deberíamos seguir los siguientes pasos

1. Ir al menú de catalogs (el simbolo que tiene un triangulo, circulo y cuadrado)
2. Crear un nuevo catalog con el nombre que queramos
3. Dentro del catalog, crear un nuevo schema
4. Dentro del Schema crear un nuevo Volume
5. Copiar la ruta que nos da el Volume y utilizarla para escribir las tablas.

También podemos crear subdirectorios para tener una mejor jerarquía de ficheros donde poder ir almacenando ficheros como mejor nos venga

In [0]:
movie_good_score.write.mode("overwrite").partitionBy("Year").save("/Volumes/prueba/test/file_clean")


In [0]:

lectura = spark.read.load("/Volumes/prueba/test/file_clean").filter(col("Year") == 2004 )

lectura.show()

+--------+----+--------------------+------------------+--------+
|Movie_ID|Year|                Name|         avg_score|Category|
+--------+----+--------------------+------------------+--------+
|     762|2004|End of the Centur...| 3.941101152368758|  Medium|
|    1865|2004|Eternal Sunshine ...| 3.721366559949237|  Medium|
|    2019|2004|    Samurai Champloo| 4.395629238884703|    High|
|    3917|2004|        Garden State|3.7062660182690412|  Medium|
|    4149|2004|Scooby-Doo 2: Mon...| 3.320169407623343|  Medium|
|     167|2004|          The Chorus| 4.021446194723857|    High|
|     846|2004|       Enduring Love| 2.691657054978854|  Medium|
|     897|2004| Bride and Prejudice| 3.291841262016268|  Medium|
|    1607|2004|Superbabies: Baby...|2.7749719416386083|  Medium|
|    2643|2004|            Carolina| 3.623326572008114|  Medium|
|    3713|2004|                 Saw|3.6169835001683657|  Medium|
|    3890|2004|Confessions of a ...|2.9332822619959606|  Medium|
|    3966|2004| Friday Ni

### spark.sql

Quería hacer especial mención a esta parte ya que, para todos aquellos que solo sepan sql, pueden utilizar spark sin ningún problema mediante el comando spark.sql(). Dentro de esos parentesis tenemos que poner la query de sql que nos interese y el nombre de la tabla, pero no podemos poner el nombre de la tabla sin más, antes tendremos que materializarla como vista, ya sea permanente o temporal, siendo esta última opción la más común.

**Ejemplo:**

Imaginemos nuestro dataframe movie. Podríamos materializarlo con movie.createOrReplaceTempView(viewName), siendo viewName un string con el que nos referiremos a nuestra tabla de ahora en adelante, por ejemplo "movie"

Una vez tenemos materializada nuestra tabla, podemos hacer spark.sql("SELECT Movie_ID FROM movie").show() para que nos mostrase el resultado de la query

### Contacto

Por último, indicar una forma de contactarme. En este caso dejaré mi id de telegram @menrric. Por ahí podeis preguntarme dudas, ya sea sobre Big Data, la empresa en la que estoy (Capgemini), spark en todas sus formas, documentación, libros, etc.

Gracias por haber asistido al taller :)